# omcwa showcase

omcwa loads OpenMovement CWA recordings from AX3 and AX6 devices, applies
omconvert-compatible auto-calibration, and resamples to a uniform sample rate.
Defaults match omgui: auto-calibrate on the full file, then keep the file
default rate unless you pass an explicit `sample_rate_hz`.

Install showcase dependencies before running this notebook:

```bash
uv sync --group showcase
# or
pip install -e ".[showcase]"
```

Set `CWA_PATH` below to your `.cwa` file before running the rest of the notebook.

In [1]:
from pathlib import Path

import numpy as np

from omcwa import (
    CalibratedRecording,
    Calibration,
    OmConvertCalibrate,
    OmConvertResample,
    ProcessedRecording,
    UniformRecording,
    load_cwa,
    open_cwa,
    process_cwa,
)
from omcwa.slice import slice_recording

# Path to your Axivity .cwa recording - edit before running.
CWA_PATH = Path("/Users/artem/Work/n8-mobgap-v2/omcwa/tests/fixtures/omgui_test.cwa")

if not CWA_PATH.is_file():
    raise FileNotFoundError(f"CWA file not found: {CWA_PATH.resolve()}")

print(f"Using CWA: {CWA_PATH.resolve()}")

Using CWA: /Users/artem/Work/n8-mobgap-v2/omcwa/tests/fixtures/omgui_test.cwa


## Default `process_cwa`

One call runs auto-calibration then resampling. With `sample_rate_hz=0` (the
default), output stays at the file default rate. Short recordings often fail
to converge: inspect `success` and `error_code` (identity fallback still
keeps the pipeline usable).

In [2]:
default_out = process_cwa(CWA_PATH)

print(f"sample_rate_hz: {default_out.sample_rate_hz}")
print(f"acc shape: {default_out.acc.shape}")
print(f"gyr present: {default_out.gyr is not None}")

if default_out.gyr is not None:
    print(f"gyr shape: {default_out.gyr.shape}")
print(
    f"calibration success: {default_out.calibration.success}, "
    f"error_code: {default_out.calibration.error_code}"
)

if default_out.valid is not None:
    print(f"valid fraction: {default_out.valid.mean():.3f}")

if default_out.clipped is not None:
    print(f"clipped fraction: {default_out.clipped.mean():.3f}")

sample_rate_hz: 100.0
acc shape: (337650, 3)
gyr present: True
gyr shape: (337650, 3)
calibration success: False, error_code: -2
valid fraction: 1.000
clipped fraction: 0.000


## Load uncalibrated samples with `load_cwa` or `open_cwa`

`load_cwa` (or `open_cwa(path).materialize()`) returns a `UniformRecording`:
file-rate float64 samples with identity calibration. Temperature is available
here (used during accel calibration) but not on `ProcessedRecording`. AX6
includes gyro, AX3 leaves `gyr` as `None`.

In [3]:
uniform = load_cwa(CWA_PATH)
# equivalent: uniform = open_cwa(CWA_PATH).materialize()

print(f"acc shape: {uniform.acc.shape}")
print(f"temp available: {uniform.temp is not None}")

if uniform.temp is not None:
    print(f"temp shape: {uniform.temp.shape}, mean: {uniform.temp.mean():.2f} C")

print(f"gyr present: {uniform.gyr is not None}")
print(f"file default rate: {uniform.metadata.get('sample_rate_hz')} Hz")

t0 = float(uniform.time[0])
window_stop = t0 + 5.0
sliced = slice_recording(uniform, start=t0, stop=window_stop)
print(f"slice [{t0:.1f}, {window_stop:.1f}) -> {len(sliced.time)} samples")

acc shape: (337650, 3)
temp available: True
temp shape: (337650,), mean: 17.40 C
gyr present: True
file default rate: 100.0 Hz
slice [1754565105.0, 1754565110.0) -> 500 samples


## Explicit rate, time window, and omconvert backends

Pass `sample_rate_hz` to target a different rate. `time_range` trims output
after processing, while auto-calibration still uses the full on-disk file.
You can also pass configured `OmConvertCalibrate` / `OmConvertResample`
instances when you need non-default omconvert parameters.

In [4]:
windowed = process_cwa(
    CWA_PATH,
    sample_rate_hz=100.0,
    time_range=(t0, t0 + 10.0),
    calibrate_fn=OmConvertCalibrate(stationary_time=10.0),
    resample_fn=OmConvertResample(sample_rate_hz=100.0),
)

print(f"sample_rate_hz: {windowed.sample_rate_hz}")
print(f"acc shape: {windowed.acc.shape}")
print(f"time span: {windowed.time[0]:.1f} to {windowed.time[-1]:.1f}")

sample_rate_hz: 100.0
acc shape: (1000, 3)
time span: 1754565105.0 to 1754565115.0


## Skip pipeline stages

Pass `None` to bypass auto-calibration (identity) or resampling (keep the
calibrated grid as-is).

In [5]:
identity_cal = process_cwa(CWA_PATH, calibrate_fn=None)
no_resample = process_cwa(CWA_PATH, resample_fn=None)

print(
    f"skip calibrate -> success={identity_cal.calibration.success}, "
    f"rate={identity_cal.sample_rate_hz}"
)
print(
    f"skip resample -> success={no_resample.calibration.success}, "
    f"rate={no_resample.sample_rate_hz}, "
    f"valid set={no_resample.valid is not None}"
)

skip calibrate -> success=True, rate=100.0
skip resample -> success=False, rate=100.0, valid set=True


## Custom calibration and resampling hooks

Replace a stage with a callable matching `CalibrateFn` or `ResampleFn`. The
hooks below are deliberately tiny: a linear scale/offset calibrator and a
`numpy.interp` resampler. Custom hooks own their own target rate.

In [6]:
def simple_calibrate(uniform: UniformRecording) -> CalibratedRecording:
    scale = np.array([1.01, 0.99, 1.0], dtype=np.float64)
    offset = np.array([0.01, -0.01, 0.0], dtype=np.float64)
    acc = uniform.acc * scale + offset
    calibration = Calibration(
        scale=scale,
        offset=offset,
        temp_offset=np.zeros(3, dtype=np.float64),
        ref_temp=25.0,
        error_code=0,
        success=True,
    )
    return CalibratedRecording(
        time=uniform.time,
        acc=acc,
        gyr=uniform.gyr,
        temp=uniform.temp,
        calibration=calibration,
        metadata=dict(uniform.metadata),
        path=uniform.path,
    )


def simple_resample(calibrated: CalibratedRecording) -> ProcessedRecording:
    target_hz = 50.0
    t_start = float(calibrated.time[0])
    t_stop = float(calibrated.time[-1])
    n = int((t_stop - t_start) * target_hz) + 1
    new_time = np.linspace(t_start, t_stop, n, dtype=np.float64)

    def interp_channels(data: np.ndarray) -> np.ndarray:
        return np.column_stack(
            [
                np.interp(new_time, calibrated.time, data[:, axis])
                for axis in range(data.shape[1])
            ]
        )

    gyr = None
    if calibrated.gyr is not None:
        gyr = interp_channels(calibrated.gyr)

    return ProcessedRecording(
        sample_rate_hz=target_hz,
        time=new_time,
        acc=interp_channels(calibrated.acc),
        gyr=gyr,
        calibration=calibrated.calibration,
        metadata=dict(calibrated.metadata),
    )


custom_out = process_cwa(
    CWA_PATH,
    calibrate_fn=simple_calibrate,
    resample_fn=simple_resample,
)

print(f"custom sample_rate_hz: {custom_out.sample_rate_hz}")
print(f"custom acc shape: {custom_out.acc.shape}")
print(f"custom calibration success: {custom_out.calibration.success}")

custom sample_rate_hz: 50.0
custom acc shape: (168825, 3)
custom calibration success: True


## Production use

Most workflows should keep the default omconvert backends (or call
`process_cwa` without custom hooks). Custom callables are useful for research
pipelines or alternative calibration and resampling strategies. Always
inspect `calibration.success` and `error_code` when auto-calibration may
fail on short recordings.